# 02 — Bronze → Silver

Limpieza con **DuckDB y SQL**, según `doc.md` (secciones 2.3, 3.3 y 3.6).
Ejecutar después de `01_generar_datos.ipynb`. Se leen las cinco fuentes de Bronze,
se guardan las tablas válidas en Parquet y los registros rechazados en CSV de cuarentena.
Bronze permanece sin modificaciones; aquí no se generan tablas Gold.

**Criterios de limpieza:**
- Normalizar espacios y tipos; conservar importes como `DECIMAL(12,2)` sin redondear entradas inválidas.
- Rechazar campos obligatorios vacíos, tipos inválidos, valores fuera del dominio y referencias inexistentes.
- Conservar una fila de cada duplicado exacto normalizado. Si un ID identifica contenidos diferentes, rechazar todas sus versiones; no elegir arbitrariamente una.
- Rechazar clientes distintos que comparten un correo normalizado.
- Validar una compra completa: cualquier línea inválida (excepto copias duplicadas), ausencia de líneas, producto repetido o total inconsistente impide aceptar el encabezado y sus líneas.

Cuarentena conserva los valores originales, el archivo, el número de registro CSV (incluyendo el encabezado) y los motivos. Se pueden registrar varios motivos para una misma fila.

In [1]:
from pathlib import Path
import csv
import hashlib
import duckdb

SCHEMAS = {
    "clientes": {"id": "INTEGER", "nombre": "VARCHAR", "correo": "VARCHAR", "fecha_registro": "DATE"},
    "productos": {"id": "INTEGER", "nombre": "VARCHAR", "categoria": "VARCHAR", "precio_normal": "DECIMAL(12,2)", "precio_en_puntos": "INTEGER"},
    "restaurantes": {"id": "INTEGER", "nombre": "VARCHAR", "zona": "VARCHAR"},
    "compras": {"id": "INTEGER", "cliente_id": "INTEGER", "restaurante_id": "INTEGER", "fecha_hora": "TIMESTAMP", "monto_total": "DECIMAL(12,2)", "forma_pago": "VARCHAR"},
    "lineas_compra": {"id": "INTEGER", "compra_id": "INTEGER", "producto_id": "INTEGER", "cantidad": "INTEGER", "precio_unitario": "DECIMAL(12,2)", "subtotal": "DECIMAL(12,2)", "pagado_con_puntos": "BOOLEAN"},
}
CATEGORIES = ('hamburguesas', 'pollo', 'desayunos', 'acompañamientos', 'bebidas', 'postres', 'ensaladas')


def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"


def typed_expression(column, data_type):
    value = f"nullif(trim({column}), '')"
    if data_type == "VARCHAR":
        return f"lower({value})" if column in {"correo", "categoria", "forma_pago"} else value
    if data_type == "INTEGER":
        valid = f"regexp_full_match({value}, '[+-]?[0-9]+')"
    elif data_type.startswith("DECIMAL"):
        # DuckDB puede redondear al convertir; esta validación evita perder precisión.
        valid = f"regexp_full_match({value}, '[+-]?[0-9]+([.][0-9]{{1,2}})?')"
    elif data_type == "BOOLEAN":
        valid = f"lower({value}) IN ('true', 'false')"
    elif data_type == "DATE":
        valid = f"regexp_full_match({value}, '[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}')"
    else:
        valid = f"regexp_full_match({value}, '[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}[ T][0-9]{{2}}:[0-9]{{2}}:[0-9]{{2}}([.][0-9]{{1,6}})?')"
    return f"CASE WHEN {valid} THEN try_cast({value} AS {data_type}) END"


def reject(connection, table, reason, condition, only_candidates=True):
    source = f"candidatos_{table}" if only_candidates else f"tipos_{table}"
    connection.execute(f"""
        INSERT INTO rechazos_{table}
        SELECT t._fila_csv, ? FROM {source} t WHERE {condition}
    """, [reason])


def load_and_type(connection, bronze):
    for table, schema in SCHEMAS.items():
        path = Path(bronze) / f"{table}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"Falta {path}; ejecutar primero el notebook 01.")
        with path.open(encoding="utf-8", newline="") as stream:
            header = next(csv.reader(stream), None)
        if header != list(schema):
            raise ValueError(f"Encabezado inesperado en {path.name}: {header}")
        # Una sola hebra y lectura secuencial mantienen el orden de los registros.
        connection.execute(f"""
            CREATE TABLE bronze_{table} AS
            SELECT row_number() OVER () + 1 AS _fila_csv, *
            FROM read_csv(?, header=true, all_varchar=true, parallel=false)
        """, [str(path)])
        columns = ', '.join(f"{typed_expression(name, dtype)} AS {name}" for name, dtype in schema.items())
        connection.execute(f"CREATE TABLE tipos_{table} AS SELECT _fila_csv, {columns} FROM bronze_{table}")
        connection.execute(f"CREATE TABLE rechazos_{table} (_fila_csv BIGINT, motivo VARCHAR)")
        connection.execute(f"""
            CREATE VIEW candidatos_{table} AS
            SELECT t.* FROM tipos_{table} t
            WHERE NOT EXISTS (SELECT 1 FROM rechazos_{table} r WHERE r._fila_csv = t._fila_csv)
        """)
        for column in schema:
            empty = f"t._fila_csv IN (SELECT _fila_csv FROM bronze_{table} WHERE nullif(trim({column}), '') IS NULL)"
            reject(connection, table, f"campo_obligatorio_vacio:{column}", empty, False)
            reject(connection, table, f"tipo_invalido:{column}", f"t.{column} IS NULL AND NOT ({empty})", False)
        for column, dtype in schema.items():
            if dtype == "INTEGER":
                reject(connection, table, f"entero_no_positivo:{column}", f"t.{column} <= 0", False)


def deduplicate(connection, table):
    fields = ', '.join(SCHEMAS[table])
    reject(connection, table, "id_conflictivo", f"""t.id IN (
        SELECT id FROM (SELECT DISTINCT {fields} FROM candidatos_{table})
        GROUP BY id HAVING count(*) > 1
    )""")
    reject(connection, table, "duplicado_exacto", f"""t._fila_csv IN (
        SELECT _fila_csv FROM (
            SELECT _fila_csv, row_number() OVER (PARTITION BY id ORDER BY _fila_csv) AS posicion
            FROM candidatos_{table}
        ) WHERE posicion > 1
    )""")

## Reglas de negocio e integridad

Primero se validan los valores y se resuelven duplicados; después, las relaciones en
el orden clientes/productos/restaurantes → compras → líneas. Finalmente se concilian
las compras completas y se propagan los rechazos a sus detalles.

Los canjes requieren precio y subtotal cero. Las líneas monetarias tienen precio positivo;
el subtotal debe ser `cantidad × precio_unitario`. No se exige que el precio histórico
cobrado coincida con el catálogo actual. Una compra compuesta solo por canjes puede tener
total cero. Las fechas deben ser válidas, posteriores al registro del cliente y dentro
del horario `[06:00, 23:00)`. No se limita esta limpieza a la ventana de análisis de ML.

In [2]:
def clean_bronze(bronze):
    connection = duckdb.connect(":memory:")
    connection.execute("SET threads = 1")
    try:
        load_and_type(connection, bronze)
        category_values = ', '.join(sql_text(category) for category in CATEGORIES)
        rules = {
            "productos": [
                ("categoria_invalida", f"t.categoria NOT IN ({category_values})"),
                ("precio_normal_no_positivo", "t.precio_normal <= 0"),
            ],
            "compras": [
                ("monto_total_negativo", "t.monto_total < 0"),
                ("forma_pago_invalida", "t.forma_pago NOT IN ('efectivo', 'tarjeta')"),
                ("hora_fuera_de_rango", "cast(t.fecha_hora AS TIME) < TIME '06:00:00' OR cast(t.fecha_hora AS TIME) >= TIME '23:00:00'"),
            ],
            "lineas_compra": [
                ("precio_unitario_negativo", "t.precio_unitario < 0"),
                ("subtotal_negativo", "t.subtotal < 0"),
                ("subtotal_inconsistente", "t.subtotal != t.cantidad::BIGINT * t.precio_unitario"),
                ("canje_con_importe_monetario", "t.pagado_con_puntos AND (t.precio_unitario != 0 OR t.subtotal != 0)"),
                ("linea_monetaria_sin_precio", "NOT t.pagado_con_puntos AND t.precio_unitario <= 0"),
            ],
        }
        for table, checks in rules.items():
            for reason, condition in checks:
                reject(connection, table, reason, condition, False)
        for table in SCHEMAS:
            deduplicate(connection, table)
        reject(connection, "clientes", "correo_compartido", """t.correo IN (
            SELECT correo FROM candidatos_clientes GROUP BY correo HAVING count(*) > 1
        )""")
        reject(connection, "compras", "cliente_inexistente_o_rechazado", "t.cliente_id NOT IN (SELECT id FROM candidatos_clientes)")
        reject(connection, "compras", "restaurante_inexistente_o_rechazado", "t.restaurante_id NOT IN (SELECT id FROM candidatos_restaurantes)")
        reject(connection, "compras", "compra_anterior_al_registro", """EXISTS (
            SELECT 1 FROM candidatos_clientes c
            WHERE c.id = t.cliente_id AND cast(t.fecha_hora AS DATE) <= c.fecha_registro
        )""")
        reject(connection, "lineas_compra", "compra_inexistente_o_rechazada", "t.compra_id NOT IN (SELECT id FROM candidatos_compras)")
        reject(connection, "lineas_compra", "producto_inexistente_o_rechazado", "t.producto_id NOT IN (SELECT id FROM candidatos_productos)")
        # Una línea rechazada invalida la factura completa, salvo una copia exacta.
        reject(connection, "compras", "contiene_linea_rechazada", """EXISTS (
            SELECT 1 FROM tipos_lineas_compra l JOIN rechazos_lineas_compra r USING (_fila_csv)
            WHERE l.compra_id = t.id AND r.motivo != 'duplicado_exacto'
        )""")
        reject(connection, "compras", "sin_lineas_validas", "NOT EXISTS (SELECT 1 FROM candidatos_lineas_compra l WHERE l.compra_id = t.id)")
        reject(connection, "compras", "detalle_fuera_de_granularidad", """t.id IN (
            SELECT compra_id FROM candidatos_lineas_compra GROUP BY compra_id
            HAVING count(*) > 4 OR count(*) != count(DISTINCT producto_id)
        )""")
        reject(connection, "compras", "monto_total_inconsistente", """t.monto_total != (
            SELECT sum(l.subtotal) FROM candidatos_lineas_compra l WHERE l.compra_id = t.id
        )""")
        reject(connection, "lineas_compra", "compra_inexistente_o_rechazada", "t.compra_id NOT IN (SELECT id FROM candidatos_compras)")
        for table, schema in SCHEMAS.items():
            fields = ', '.join(schema)
            connection.execute(f"CREATE TABLE silver_{table} AS SELECT {fields} FROM candidatos_{table} ORDER BY id")
        return connection
    except Exception:
        connection.close()
        raise


def quality_summary(connection):
    rows = []
    for table in SCHEMAS:
        total = connection.execute(f"SELECT count(*) FROM bronze_{table}").fetchone()[0]
        accepted = connection.execute(f"SELECT count(*) FROM silver_{table}").fetchone()[0]
        rejected = connection.execute(f"SELECT count(DISTINCT _fila_csv) FROM rechazos_{table}").fetchone()[0]
        assert total == accepted + rejected, f"Filas sin clasificar o contadas dos veces: {table}"
        rows.append({"tabla": table, "bronze": total, "silver": accepted, "cuarentena": rejected})
    return rows


def validate_silver(connection):
    for table, schema in SCHEMAS.items():
        null_checks = ' OR '.join(f"{name} IS NULL" for name in schema)
        assert connection.execute(f"SELECT count(*) FROM silver_{table} WHERE {null_checks}").fetchone()[0] == 0
        assert connection.execute(f"SELECT count(*) - count(DISTINCT id) FROM silver_{table}").fetchone()[0] == 0
        actual = {row[0]: row[1] for row in connection.execute(f"DESCRIBE silver_{table}").fetchall()}
        assert actual == schema, (table, actual)
    for child, column, parent in [
        ('compras', 'cliente_id', 'clientes'), ('compras', 'restaurante_id', 'restaurantes'),
        ('lineas_compra', 'compra_id', 'compras'), ('lineas_compra', 'producto_id', 'productos'),
    ]:
        assert connection.execute(f"SELECT count(*) FROM silver_{child} WHERE {column} NOT IN (SELECT id FROM silver_{parent})").fetchone()[0] == 0
    assert connection.execute("SELECT count(*) - count(DISTINCT correo) FROM silver_clientes").fetchone()[0] == 0
    assert connection.execute("""
        SELECT count(*) FROM silver_compras c
        LEFT JOIN (SELECT compra_id, count(*) AS n, count(DISTINCT producto_id) AS productos,
                          sum(subtotal) AS total FROM silver_lineas_compra GROUP BY compra_id) l
        ON c.id = l.compra_id
        WHERE l.compra_id IS NULL OR l.n NOT BETWEEN 1 AND 4 OR l.productos != l.n OR l.total != c.monto_total
    """).fetchone()[0] == 0
    assert connection.execute("""
        SELECT count(*) FROM silver_lineas_compra
        WHERE cantidad < 1 OR precio_unitario < 0 OR subtotal < 0
           OR subtotal != cantidad::BIGINT * precio_unitario
           OR (pagado_con_puntos AND (precio_unitario != 0 OR subtotal != 0))
           OR (NOT pagado_con_puntos AND precio_unitario <= 0)
    """).fetchone()[0] == 0
    return quality_summary(connection)


def export_silver(connection, silver_dir, quarantine_dir):
    validate_silver(connection)
    silver_dir, quarantine_dir = Path(silver_dir), Path(quarantine_dir)
    silver_dir.mkdir(parents=True, exist_ok=True)
    quarantine_dir.mkdir(parents=True, exist_ok=True)
    for table, schema in SCHEMAS.items():
        parquet = silver_dir / f"{table}.parquet"
        quarantine = quarantine_dir / f"{table}.csv"
        connection.execute(f"COPY (SELECT * FROM silver_{table} ORDER BY id) TO {sql_text(parquet)} (FORMAT PARQUET, COMPRESSION ZSTD)")
        raw_columns = ', '.join(f"b.{name}" for name in schema)
        connection.execute(f"""
            COPY (
                SELECT {sql_text(table + '.csv')} AS archivo_origen, b._fila_csv AS fila_csv,
                       {raw_columns}, r.motivos
                FROM bronze_{table} b
                JOIN (SELECT _fila_csv, string_agg(DISTINCT motivo, ' | ' ORDER BY motivo) AS motivos
                      FROM rechazos_{table} GROUP BY _fila_csv) r USING (_fila_csv)
                ORDER BY b._fila_csv
            ) TO {sql_text(quarantine)} (FORMAT CSV, HEADER true)
        """)
        # Verificar contenido y tipos después de leer el Parquet escrito.
        connection.execute(f"CREATE OR REPLACE VIEW exportado_{table} AS SELECT * FROM read_parquet({sql_text(parquet)})")
        types = {row[0]: row[1] for row in connection.execute(f"DESCRIBE exportado_{table}").fetchall()}
        assert types == schema
        difference = connection.execute(f"""
            SELECT count(*) FROM (
                (SELECT * FROM silver_{table} EXCEPT ALL SELECT * FROM exportado_{table})
                UNION ALL
                (SELECT * FROM exportado_{table} EXCEPT ALL SELECT * FROM silver_{table})
            )
        """).fetchone()[0]
        assert difference == 0, f"El Parquet cambió los datos: {table}"
        with quarantine.open(encoding="utf-8", newline="") as stream:
            rows = list(csv.DictReader(stream))
        expected = connection.execute(f"SELECT count(DISTINCT _fila_csv) FROM rechazos_{table}").fetchone()[0]
        assert len(rows) == expected and all(row['motivos'] for row in rows)

## Ejecutar limpieza y revisar resultados

El motor trabaja en memoria. Todas las reglas se calculan de nuevo al ejecutar el notebook,
por lo que no se acumulan rechazos de ejecuciones anteriores. Los cinco CSV originales
se protegen mediante una comparación de sus huellas SHA-256 antes y después del proceso.

In [3]:
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "doc.md").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Ejecutar desde la raíz del proyecto o notebooks/.")
BRONZE = PROJECT_ROOT / "data" / "bronze"
SILVER = PROJECT_ROOT / "data" / "silver"
QUARANTINE = PROJECT_ROOT / "data" / "cuarentena"

def bronze_hashes():
    return {table: hashlib.sha256((BRONZE / f"{table}.csv").read_bytes()).hexdigest() for table in SCHEMAS}

before = bronze_hashes()
if "connection" in globals():
    connection.close()
connection = clean_bronze(BRONZE)
summary = validate_silver(connection)
print(f"DuckDB {duckdb.__version__}")
print(f"{'Tabla':20s} {'Bronze':>8s} {'Silver':>8s} {'Cuarentena':>12s}")
for row in summary:
    print(f"{row['tabla']:20s} {row['bronze']:8,d} {row['silver']:8,d} {row['cuarentena']:12,d}")
print("Todas las filas están contabilizadas: Bronze = Silver + cuarentena.")

DuckDB 1.5.5
Tabla                  Bronze   Silver   Cuarentena
clientes                1,002    1,000            2
productos                  30       28            2
restaurantes               12       10            2
compras                20,007   20,000            7
lineas_compra          49,532   49,525            7
Todas las filas están contabilizadas: Bronze = Silver + cuarentena.


In [4]:
# Conteos por motivo: una fila puede tener más de un motivo.
for table in SCHEMAS:
    reasons = connection.execute(f"SELECT motivo, count(DISTINCT _fila_csv) AS filas FROM rechazos_{table} GROUP BY motivo ORDER BY motivo").fetchall()
    print(f"\n{table}:")
    for reason, count in reasons:
        print(f"  {reason}: {count}")


clientes:
  duplicado_exacto: 2

productos:
  duplicado_exacto: 2

restaurantes:
  duplicado_exacto: 2

compras:
  contiene_linea_rechazada: 1
  duplicado_exacto: 5
  monto_total_negativo: 1

lineas_compra:
  duplicado_exacto: 5
  linea_monetaria_sin_precio: 1
  precio_unitario_negativo: 1
  producto_inexistente_o_rechazado: 1
  subtotal_negativo: 1


## Exportar y verificar

Se escriben `data/silver/<tabla>.parquet` y `data/cuarentena/<tabla>.csv` para cada una
de las cinco tablas. La reejecución reemplaza estos mismos diez archivos. Incluso si no
hay rechazos, el CSV de cuarentena conserva su encabezado.

Además de comprobar claves y montos, se releen los Parquet para verificar que fechas,
enteros, decimales y booleanos mantengan sus tipos y valores.

In [5]:
export_silver(connection, SILVER, QUARANTINE)
assert before == bronze_hashes(), "Bronze fue modificado"
for table in SCHEMAS:
    print(f"{table}: Parquet y cuarentena verificados.")
print("Bronze permanece intacto. Silver está listo para el notebook 03.")
connection.close()

clientes: Parquet y cuarentena verificados.
productos: Parquet y cuarentena verificados.
restaurantes: Parquet y cuarentena verificados.
compras: Parquet y cuarentena verificados.
lineas_compra: Parquet y cuarentena verificados.
Bronze permanece intacto. Silver está listo para el notebook 03.


## Resultado esperado con el notebook 01 actual

| Tabla | Bronze | Silver | Cuarentena |
|---|---:|---:|---:|
| clientes | 1,002 | 1,000 | 2 |
| productos | 30 | 28 | 2 |
| restaurantes | 12 | 10 | 2 |
| compras | 20,007 | 20,000 | 7 |
| lineas_compra | 49,532 | 49,525 | 7 |

Los 20 registros rechazados corresponden a 16 duplicados, la compra negativa y su línea,
y la compra con producto inexistente y su línea. El encabezado de esta última se rechaza
porque su detalle no es válido. Los conteos son una referencia de la semilla actual;
el algoritmo no depende de estos IDs ni de estos volúmenes.

La siguiente fase calculará perfiles y ventas en Gold usando únicamente los meses 1–2.
Este notebook no entrena modelos ni aplica multiplicadores.